# 🧠 AI Engineering Assignment #01 — Transformer & GPT From Scratch

> **Student**: Zaid Bin Naveed  
> **Repository**: [zaidbinnaveed/AI-Engineering-Assignment-01](https://github.com/zaidbinnaveed/AI-Engineering-Assignment-01)  
> **Hardware**: Google Colab GPU (Tesla T4)  

This notebook trains a complete causal language model (GPT-style Pre-LN Transformer) from scratch on **Tiny Shakespeare** without loading any pretrained models.

### Pipeline Overview:
1. **Environment Setup & GPU Check** (`nvidia-smi`)
2. **Repository Clone & Package Installation** (`pip install -e .`)
3. **324+ Test Verification** (`pytest`)
4. **Data Preparation & Cryptographic Verification** (Tiny Shakespeare SHA256)
5. **Milestone M4 Overfit Gate** (Confirms zero-loss convergence before long training)
6. **Phase 6 Training** (1,500 steps baseline GPT on GPU)
7. **Evidence Figures**: F06 (Loss Curves), F07 (Trained Attention Maps), F08 (Generated Samples)
8. **Phase 7 Controlled Ablations**: Depth, Heads, and Context window (F09)
9. **Artifact Export & Download** (`trained_artifacts.zip`)

--- 
## 1. Verify GPU Acceleration
*Make sure to set **Runtime > Change runtime type > T4 GPU**.*

In [ ]:
!nvidia-smi

import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Running on CPU. Please enable GPU in Colab Runtime settings for maximum speed.")

--- 
## 2. Clone Repository & Install Package

In [ ]:
import os
import sys

# Clone or pull latest updates
if not os.path.exists("AI-Engineering-Assignment-01") and not os.path.exists("src"):
    !git clone https://github.com/zaidbinnaveed/AI-Engineering-Assignment-01.git
    %cd AI-Engineering-Assignment-01
elif os.path.exists("AI-Engineering-Assignment-01"):
    %cd AI-Engineering-Assignment-01
    !git pull

# Ensure project root is in PYTHONPATH
%env PYTHONPATH=.
if "." not in sys.path:
    sys.path.insert(0, ".")

# Install project in editable development mode
!pip install -e .
!pip install matplotlib pytest

--- 
## 3. Run PyTest Unit & Integration Suite
Verifies all 324 unit tests across attention math, positional encoding, decoder block, GPT model, shifted targets, tokenizer, and checkpointing.

In [ ]:
!pytest

--- 
## 4. Prepare Tiny Shakespeare Dataset
Downloads and verifies raw Tiny Shakespeare text (SHA256: `86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed`).
Splits into 90% train / 10% validation and derives the 65-character vocabulary strictly on training data.

In [ ]:
from zaid_lm.dataset import prepare_dataset

splits = prepare_dataset("data")
print(f"Total Training Tokens:   {len(splits.train_tokens):,}")
print(f"Total Validation Tokens: {len(splits.val_tokens):,}")
print(f"Vocabulary Size:         {splits.tokenizer.vocab_size} unique characters")
print(f"Sample Characters:       {splits.tokenizer.vocabulary[:20]}...")

--- 
## 5. Milestone M4: Tiny-Subset Overfit Gate
Mandatory sanity gate: trains a small GPT on a single fixed sequence to prove causal masking, shifted next-token targets ($x[t] \to y[t]=x[t+1]$), and AdamW backpropagation drive loss to near-zero before long training.

In [ ]:
!python experiments/tiny_overfit.py

--- 
## 6. Phase 6: Train Baseline GPT Model on GPU
Trains the baseline compact GPT model (809,984 parameters, 4 layers, 4 heads, $d_{\text{model}}=128$, context length 128).
On a Colab T4 GPU, 1,500 steps executes in **~30 to 45 seconds**!

In [ ]:
!python -m zaid_lm.train --config configs/baseline.json --device cuda --steps 1500 --batch-size 32 --eval-interval 100

--- 
## 7. Generate Phase 6 Evidence Figures (F06, F07, F08)
- **F06**: Real train vs validation loss & perplexity curves
- **F07**: Attention heatmaps extracted from the real trained checkpoint
- **F08**: Authentic generated samples from 5 fixed prompts (ROMEO, JULIET, HAMLET, etc.)

In [ ]:
# Generate Figure F06 (Loss Curves)
!python -m experiments.visualize_loss_curves

# Generate Figure F07 (Trained Attention Maps)
!python -m experiments.visualize_trained_attention

# Generate Figure F08 (5 Authentic Generated Samples)
!python -m experiments.generate_samples

### Display Generated Figures Directly in Notebook

In [ ]:
from IPython.display import Image, display

print("=== F06: Loss & Perplexity Curves ===")
display(Image("report/evidence/F06_loss.png"))

print("=== F07: Trained Attention Heatmaps ===")
display(Image("report/evidence/F07_trained_attention.png"))

print("=== F08: Authentic Generated Samples ===")
display(Image("report/evidence/F08_samples.png"))

--- 
## 8. Phase 7: Controlled Architectural Ablations (Figure F09)
Executes 3 single-factor controlled perturbations against baseline:
1. **Depth**: 2 Layers vs 4 Layers
2. **Heads**: 2 Attention Heads vs 4 Heads
3. **Context**: 64 tokens vs 128 tokens

On GPU, all 4 models train in ~2 minutes!

In [ ]:
!python -m experiments.run_ablations --device cuda --steps 600

display(Image("report/evidence/F09_ablations.png"))

--- 
## 9. Export & Download All Artifacts
Packages the trained checkpoint (`best_model.pt`), generated figures (F01–F09), and logs into a zip file for local saving.

In [ ]:
!zip -r trained_artifacts.zip checkpoints/ outputs/ report/evidence/ data/provenance.json

from google.colab import files

files.download("trained_artifacts.zip")